In [1]:
from __future__ import annotations

import os
import re
import json
import math
import random
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import weight_norm
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor, get_cosine_schedule_with_warmup
from accelerate import Accelerator

In [2]:
import sys
sys.path.append('/workspace/MedVQA-Classification/src')

In [3]:
from data.dataloaders import *

In [4]:
from __future__ import annotations

from dataclasses import dataclass, fields
from pathlib import Path
from typing import Optional, Any, Dict

import yaml


@dataclass
class TrainConfig:
    dataset_name: str = "flaviagiammarino/vqa-rad"
    model_name: str = "flaviagiammarino/pubmed-clip-vit-base-patch32"
    model_type: str = "pubmedclip_ban"

    output_dir: str = "runs/pubmedclip_ban"
    seed: int = 42

    epochs: int = 40
    batch_size: int = 16
    eval_batch_size: int = 64
    num_workers: int = 2
    max_length: int = 77

    num_hid: int = 1536
    glimpse: int = 4
    freeze_clip: bool = True

    lr_head: float = 1e-4
    lr_clip: float = 1e-6
    weight_decay: float = 1e-4
    warmup_ratio: float = 0.05
    type_loss_weight: float = 0.2
    class_weight_power: float = 0.5
    mixed_precision: str = "no"
    grad_clip: float = 1.0

    min_answer_freq: int = 1
    max_answers: Optional[int] = None
    answer_vocab_source: str = "train_eval"
    filter_eval_unseen_train_answers: bool = False

    filter_invalid_count_answers: bool = False
    count_answer_max_num: Optional[int] = None

    eval_with_type_mask: bool = True


def load_config(path: str | Path, overrides: Optional[Dict[str, Any]] = None) -> TrainConfig:
    path = Path(path)
    data = yaml.safe_load(path.read_text(encoding="utf-8")) or {}

    if overrides:
        for key, value in overrides.items():
            if value is not None:
                data[key] = value

    valid_fields = {f.name for f in fields(TrainConfig)}
    unknown = set(data) - valid_fields
    if unknown:
        raise ValueError(f"Unknown config keys: {sorted(unknown)}")

    return TrainConfig(**data)


In [5]:
class FCNet(nn.Module):
    def __init__(self, dims: List[int], act: str = "ReLU", dropout: float = 0.2):
        super().__init__()
        layers = []
        for i in range(len(dims) - 1):
            layers.append(weight_norm(nn.Linear(dims[i], dims[i + 1],bias=False), name="weight", dim=None))
            if act:
                layers.append(getattr(nn, act)())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        self.main = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.main(x)


class BCNet(nn.Module):
    def __init__(self, v_dim: int, q_dim: int, h_dim: int, h_out: Optional[int] = None,
                 dropout: Tuple[float, float] = (0.2, 0.5), k: int = 3):
        super().__init__()
        self.h_dim = h_dim
        self.h_out = h_out
        self.k = k
        self.v_net = FCNet([v_dim, h_dim * k], act="ReLU", dropout=dropout[0])
        self.q_net = FCNet([q_dim, h_dim * k], act="ReLU", dropout=dropout[0])
        self.dropout = nn.Dropout(dropout[1])
        self.p_net = nn.AvgPool1d(k, stride=k) if k > 1 else None

        if h_out is not None:
            self.h_mat = nn.Parameter(torch.randn(1, h_out, 1, h_dim * k) * 0.01)
            self.h_bias = nn.Parameter(torch.zeros(1, h_out, 1, 1))

    def forward(self, v: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
        v_ = self.dropout(self.v_net(v))
        q_ = self.q_net(q)
        if self.h_out is None:
            return torch.einsum("bvk,bqk->bvqk", v_, q_)
        return torch.einsum("ghyk,bvk,bqk->bhvq", self.h_mat, v_, q_) + self.h_bias

    def forward_with_weights(self, v: torch.Tensor, q: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
        v_ = self.v_net(v)
        q_ = self.q_net(q)
        logits = torch.einsum("bvk,bvq,bqk->bk", v_, w, q_)
        if self.k > 1:
            logits = self.p_net(logits.unsqueeze(1)).squeeze(1) * self.k
        return logits


class BiAttention(nn.Module):
    def __init__(self, v_dim: int, q_dim: int, h_dim: int, glimpse: int = 4):
        super().__init__()
        self.glimpse = glimpse
        self.logits = BCNet(v_dim, q_dim, h_dim, h_out=glimpse, dropout=(0.2, 0.5), k=3)

    def forward_all(self, v: torch.Tensor, q: torch.Tensor, q_mask: Optional[torch.Tensor] = None):
        B, V, Q = v.size(0), v.size(1), q.size(1)
        logits = self.logits(v, q)
        if q_mask is not None:
            mask = q_mask[:, None, None, :].bool()
            logits = logits.masked_fill(~mask, -1e4)
        att = F.softmax(logits.reshape(B, self.glimpse, V * Q), dim=-1).reshape(B, self.glimpse, V, Q)
        return att, logits


class SimpleClassifier(nn.Module):
    def __init__(self, in_dim: int, hid_dim: int, out_dim: int, dropout: float = 0.5):
        super().__init__()
        self.main = nn.Sequential(
            weight_norm(nn.Linear(in_dim, hid_dim,bias=False), name="weight", dim=None),
            nn.ReLU(),
            nn.Dropout(dropout),
            weight_norm(nn.Linear(hid_dim, out_dim,bias=False), name="weight", dim=None),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.main(x)


class PubMedCLIPBANFixed(nn.Module):
    def __init__(
        self,
        clip_model: CLIPModel,
        num_answers: int,
        num_hid: int = 512,
        glimpse: int = 4,
        freeze_clip: bool = True,
    ):
        super().__init__()
        self.clip = clip_model
        self.freeze_clip = freeze_clip
        self.v_dim = clip_model.config.vision_config.hidden_size
        self.q_dim = clip_model.config.text_config.hidden_size
        self.glimpse = glimpse

        if freeze_clip:
            for p in self.clip.parameters():
                p.requires_grad = False

        self.v_att = BiAttention(self.v_dim, self.q_dim, num_hid, glimpse=glimpse)
        self.b_net = nn.ModuleList([
            BCNet(self.v_dim, self.q_dim, num_hid, h_out=None, dropout=(0.2, 0.5), k=3)
            for _ in range(glimpse)
        ])
        self.q_prj = nn.ModuleList([
            FCNet([num_hid, self.q_dim], act="ReLU", dropout=0.2)
            for _ in range(glimpse)
        ])

        self.answer_classifier = SimpleClassifier(self.q_dim, num_hid * 2, num_answers, dropout=0.5)
        self.type_classifier = SimpleClassifier(self.q_dim, num_hid, 2, dropout=0.3)

    def train(self, mode: bool = True):
        super().train(mode)
        if self.freeze_clip:
            self.clip.eval()
        return self

    def encode_tokens(self, pixel_values, input_ids, attention_mask):
        grad_enabled = any(p.requires_grad for p in self.clip.parameters())
        with torch.set_grad_enabled(grad_enabled):
            vision_outputs = self.clip.vision_model(pixel_values=pixel_values, return_dict=True)
            text_outputs = self.clip.text_model(
                input_ids=input_ids, attention_mask=attention_mask, return_dict=True
            )
        # ViT: bỏ CLS token; ResNet CLIP có thể cần chỉnh lại nếu dùng RN backend.
        visual_tokens = vision_outputs.last_hidden_state[:, 1:, :]
        text_tokens = text_outputs.last_hidden_state
        return visual_tokens, text_tokens

    def forward(self, pixel_values, input_ids, attention_mask):
        v, q = self.encode_tokens(pixel_values, input_ids, attention_mask)
        att, _ = self.v_att.forward_all(v, q, q_mask=attention_mask)

        for g in range(self.glimpse):
            b_emb = self.b_net[g].forward_with_weights(v, q, att[:, g])
            q = q + self.q_prj[g](b_emb).unsqueeze(1)

        mask = attention_mask.unsqueeze(-1).float()
        q_pooled = (q * mask).sum(1) / mask.sum(1).clamp_min(1.0)

        answer_logits = self.answer_classifier(q_pooled)
        type_logits = self.type_classifier(q_pooled)
        return answer_logits, type_logits, att


def mask_logits_by_predicted_type(
    logits: torch.Tensor,
    type_logits: torch.Tensor,
    label_type_ids: torch.Tensor,
    mask_value: float = -1e4,
) -> torch.Tensor:
    """
    label_type_ids: [num_answers], 0=open, 1=closed
    type_logits -> predicted type per sample.
    """
    pred_type = type_logits.argmax(dim=-1)  # [B]
    allowed = label_type_ids.unsqueeze(0).to(logits.device) == pred_type.unsqueeze(1)
    return logits.masked_fill(~allowed, mask_value)

In [6]:
from engine.evaluate import evaluate

In [7]:
from engine.trainer import train_main

In [8]:
# from collections import Counter
# # def train_main(cfg: TrainConfig):
# #     # seed_everything(cfg.seed)

# #     accelerator = Accelerator(mixed_precision=cfg.mixed_precision)
# #     if accelerator.is_main_process:
# #         os.makedirs(cfg.output_dir, exist_ok=True)
# #         with open(os.path.join(cfg.output_dir, "config.json"), "w", encoding="utf-8") as f:
# #             json.dump(asdict(cfg), f, indent=2, ensure_ascii=False)
# #     falsed_sample=Counter()
# #     processor = CLIPProcessor.from_pretrained(cfg.model_name)
# #     data_bundle = make_loaders(cfg, processor)
# #     train_loader = data_bundle["train_loader"]
# #     eval_loader = data_bundle["eval_loader"]
# #     test_loader = data_bundle["test_loader"]
# #     label2ans = data_bundle["label2ans"]
# #     ans2label = data_bundle["ans2label"]
# #     label_type_ids = data_bundle["label_type_ids"]
# #     class_weights = data_bundle["class_weights"]
# #     train_answer_set = data_bundle["train_answer_set"]
# #     filter_stats = data_bundle["filter_stats"]
# #     if accelerator.is_main_process:
# #         print(
# #             "[count-filter] "
# #             f"enabled={filter_stats['filter_invalid_count_answers']} | "
# #             f"max_num={filter_stats['count_answer_max_num']} | "
# #             f"train {filter_stats['train_original_len']} -> {filter_stats['train_filtered_len']} "
# #             f"(invalid_count={filter_stats['train_invalid_count_answer_samples']}) | "
# #             f" {filter_stats['eval_original_len']} -> {filter_stats['eval_filtered_len']} "
# #             f"(invalid_count={filter_stats['eval_invalid_count_answer_samples']})"
# #         )

# #     clip_model = CLIPModel.from_pretrained(cfg.model_name)
# #     model = PubMedCLIPBANFixed(
# #         clip_model=clip_model,
# #         num_answers=len(label2ans),
# #         num_hid=cfg.num_hid,
# #         glimpse=cfg.glimpse,
# #         freeze_clip=cfg.freeze_clip,
# #     )
# #     head_params, clip_params = [], []
# #     for n, p in model.named_parameters():
# #         if not p.requires_grad:
# #             continue
# #         if n.startswith("clip."):
# #             clip_params.append(p)
# #         else:
# #             head_params.append(p)

# #     optimizer = torch.optim.AdamW(
# #         [
# #             {"params": head_params, "lr": cfg.lr_head},
# #             {"params": clip_params, "lr": cfg.lr_clip},
# #         ],
# #         weight_decay=cfg.weight_decay,
# #     )
# #     total_steps = cfg.epochs * math.ceil(len(train_loader) / max(accelerator.num_processes, 1))
# #     warmup_steps = int(total_steps * cfg.warmup_ratio)
# #     scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# #     model, optimizer, train_loader, test_loader, scheduler = accelerator.prepare(
# #         model, optimizer, train_loader, test_loader, scheduler
# #     )
# #     class_weights = class_weights.to(accelerator.device)
# #     label_type_ids = label_type_ids.to(accelerator.device)

# #     best = -1.0
# #     best_path = os.path.join(cfg.output_dir, "best_model.pt")
# #     for epoch in range(1, cfg.epochs + 1):
# #         model.train()
# #         loss_meter = 0.0
# #         step_count = 0

# #         for batch in train_loader:
# #             optimizer.zero_grad(set_to_none=True)
# #             answer_logits, type_logits, _ = model(
# #                 pixel_values=batch["pixel_values"],
# #                 input_ids=batch["input_ids"],
# #                 attention_mask=batch["attention_mask"],
# #             )

# #             answer_loss = F.cross_entropy(
# #                 answer_logits,
# #                 batch["labels"],
# #                 weight=class_weights,
# #                 ignore_index=-100,
# #             )
# #             type_loss = F.cross_entropy(type_logits, batch["answer_type"])
# #             loss = answer_loss + cfg.type_loss_weight * type_loss

# #             accelerator.backward(loss)
# #             if cfg.grad_clip and cfg.grad_clip > 0:
# #                 accelerator.clip_grad_norm_(model.parameters(), cfg.grad_clip)
# #             optimizer.step()
# #             scheduler.step()

# #             loss_meter += accelerator.gather_for_metrics(loss.detach()).mean().item()
# #             step_count += 1
# #         metrics = evaluate(model, test_loader, accelerator, label_type_ids, cfg)
# #         falsed_sample.update(metrics['falsed_samples'])
# #         if accelerator.is_main_process:
# #             score = metrics["overall_acc"]
# #             if score > best:
# #                 best = score
# #                 unwrapped = accelerator.unwrap_model(model)
# #                 torch.save(
# #                     {
# #                         "model": unwrapped.state_dict(),
# #                         "label2ans": label2ans,
# #                         "ans2label": ans2label,
# #                         "label_type_ids": label_type_ids.detach().cpu(),
# #                         "train_answer_set": sorted(list(train_answer_set)),
# #                         "answer_vocab_source": cfg.answer_vocab_source,
# #                         "filter_stats": filter_stats,
# #                         "config": asdict(cfg),
# #                     },
# #                     best_path,
# #                 )

# #             print(
# #                 f"Epoch {epoch:03d}/{cfg.epochs} | "
# #                 f"loss={loss_meter / max(step_count, 1):.4f} | "
# #                 f"overall={metrics['overall_acc']:.4f} | "
# #                 f"open={metrics['open_acc']:.4f} | "
# #                 f"closed={metrics['closed_acc']:.4f} | "
               
# #                 f"best={best:.4f}"
# #             )

# #     accelerator.wait_for_everyone()
# #     print(falsed_sample)
# #     return best_path,falsed_sample


In [9]:
from accelerate import notebook_launcher

# cfg = TrainConfig(
#     dataset_name="flaviagiammarino/vqa-rad",
#     model_name="flaviagiammarino/pubmed-clip-vit-base-patch32",
#     output_dir="./medvqa_runs/vqa_rad_pubmedclip_ban",
#     epochs=200,
#     batch_size=32,
#     lr_head=1e-4,
#     lr_clip=1e-5,
#     freeze_clip=True,
#     answer_vocab_source="train_eval",  # repo/paper-compatible. Dùng "train" cho strict protocol.
#     mixed_precision="bf16",
# )
cfg = TrainConfig(dataset_name= '/workspace/MedVQA-Classification/datasets/slake',epochs=1)
notebook_launcher(train_main, args=(cfg,), num_processes=1)

Launching training on one GPU.


[count-filter] enabled=False | max_num=None | train 4919 -> 4919 (invalid_count=0) | 1053 -> 1053 (invalid_count=0)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Epoch 001/1 | loss=2.4587 | test_overall=0.5156 | test_open=0.5070 | test_closed=0.5288 | val_overall=0.5404 | val_open=0.5388 | val_closed=0.5427 | test_best=0.5156


In [ ]:
import os
import sys

sys.path.append('/workspace/MedVQA-Classification/src')
sys.path.append('/workspace/LLaVA-Med/')
from data.data_prep import PreDataset


t_s=PreDataset('/workspace/MedVQA-Classification/datasets/slake','train')

In [2]:
from data.data_prep import PreDataset


t_s=PreDataset('/workspace/MedVQA-Classification/datasets/slake','train')

In [3]:
t_s[0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256>,
 'question': 'What modality is used to take this image?',
 'answer': 'MRI',
 'answer_type': 'OPEN'}

In [ ]:
# # =========================
# # One-cell LLaVA-Med inference for Jupyter/IPYNB
# # Match official LLaVA-style inference more closely
# # =========================

# import os
# import sys
# import gc
# import torch
# from PIL import Image

# # -------- CONFIG --------
# LLAVA_MED_PATH = "/workspace/LLaVA-Med"
# MODEL_PATH = "microsoft/llava-med-v1.5-mistral-7b"

# # Sample từ dataset của bạn
# image = t_s[0]["image"].convert("RGB")
# question = t_s[0]["question"]

# MAX_NEW_TOKENS = 128

# # VQA/classification nên để deterministic
# DO_SAMPLE = False
# TEMPERATURE = 0.0
# TOP_P = None
# NUM_BEAMS = 1

# # True = chỉ in câu trả lời mới sinh ra
# # False = in full output giống infer_one.py cũ của bạn
# ANSWER_ONLY = True

# # True = chạy xong xóa model để cố gắng trả GPU RAM
# # False = giữ model trong RAM để infer nhiều lần nhanh hơn
# CLEANUP_AFTER_INFER = True

# LOAD_4BIT = False
# LOAD_8BIT = False
# # ------------------------

# sys.path.insert(0, LLAVA_MED_PATH)

# from llava.model.builder import load_pretrained_model
# from llava.mm_utils import (
#     get_model_name_from_path,
#     process_images,
#     tokenizer_image_token,
# )
# from llava.constants import (
#     IMAGE_TOKEN_INDEX,
#     DEFAULT_IMAGE_TOKEN,
#     DEFAULT_IM_START_TOKEN,
#     DEFAULT_IM_END_TOKEN,
# )
# from llava.conversation import conv_templates

# try:
#     from llava.utils import disable_torch_init
#     disable_torch_init()
# except Exception:
#     pass


# def show_gpu_mem(tag=""):
#     if torch.cuda.is_available():
#         allocated = torch.cuda.memory_allocated() / 1024**3
#         reserved = torch.cuda.memory_reserved() / 1024**3
#         print(f"{tag} allocated={allocated:.2f} GB | reserved={reserved:.2f} GB")


# def get_model_device(model):
#     try:
#         return model.device
#     except AttributeError:
#         return next(model.parameters()).device


# def cleanup_cuda():
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.ipc_collect()


# show_gpu_mem("[Before load]")

# # -------- LOAD MODEL --------
# model_name = get_model_name_from_path(MODEL_PATH)

# try:
#     tokenizer, model, image_processor, context_len = load_pretrained_model(
#         model_path=MODEL_PATH,
#         model_base=None,
#         model_name=model_name,
#         load_4bit=LOAD_4BIT,
#         load_8bit=LOAD_8BIT,
#         device_map="auto",
#     )
# except TypeError:
#     tokenizer, model, image_processor, context_len = load_pretrained_model(
#         model_path=MODEL_PATH,
#         model_base=None,
#         model_name=model_name,
#     )

# model.eval()
# device = get_model_device(model)

# show_gpu_mem("[After load]")

# # -------- BUILD PROMPT giống official hơn --------
# if getattr(model.config, "mm_use_im_start_end", False):
#     image_token = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN
# else:
#     image_token = DEFAULT_IMAGE_TOKEN

# prompt_question = image_token + "\n" + question

# # Auto conv_mode giống official: model_name có "mistral" -> mistral_instruct
# if "mistral" in model_name.lower() and "mistral_instruct" in conv_templates:
#     conv_mode = "mistral_instruct"
# elif "llava_v1" in conv_templates:
#     conv_mode = "llava_v1"
# elif "mistral" in conv_templates:
#     conv_mode = "mistral"
# else:
#     conv_mode = list(conv_templates.keys())[0]

# conv = conv_templates[conv_mode].copy()
# conv.append_message(conv.roles[0], prompt_question)
# conv.append_message(conv.roles[1], None)
# prompt = conv.get_prompt()

# print("Using conv_mode:", conv_mode)
# print("Question:", question)

# # -------- PREPROCESS IMAGE --------
# image_tensor = process_images(
#     [image],
#     image_processor,
#     model.config,
# )

# if isinstance(image_tensor, list):
#     image_tensor = [
#         img.to(device=device, dtype=torch.float16)
#         for img in image_tensor
#     ]
# else:
#     image_tensor = image_tensor.to(
#         device=device,
#         dtype=torch.float16,
#     )

# image_sizes = [image.size]

# input_ids = tokenizer_image_token(
#     prompt,
#     tokenizer,
#     IMAGE_TOKEN_INDEX,
#     return_tensors="pt",
# ).unsqueeze(0).to(device)

# # -------- GENERATE --------
# gen_kwargs = dict(
#     images=image_tensor,
#     image_sizes=image_sizes,
#     do_sample=DO_SAMPLE,
#     max_new_tokens=MAX_NEW_TOKENS,
#     use_cache=True,
#     num_beams=NUM_BEAMS,
# )

# if DO_SAMPLE:
#     gen_kwargs["temperature"] = TEMPERATURE
#     gen_kwargs["top_p"] = TOP_P

# with torch.inference_mode():
#     output_ids = model.generate(
#         input_ids,
#         **gen_kwargs,
#     )

# # -------- DECODE --------
# if ANSWER_ONLY and output_ids.shape[1] > input_ids.shape[1]:
#     decode_ids = output_ids[0, input_ids.shape[1]:]
# else:
#     decode_ids = output_ids[0]

# answer = tokenizer.decode(
#     decode_ids,
#     skip_special_tokens=True,
# ).strip()

# # Dọn các stop string thường gặp
# for stop in ["</s>", "###", "USER:", "ASSISTANT:"]:
#     if stop in answer:
#         answer = answer.split(stop)[0].strip()

# print("Answer:", answer)

# # -------- CLEAN TEMP TENSORS --------
# del image_tensor, input_ids, output_ids, decode_ids
# cleanup_cuda()

# show_gpu_mem("[After infer temp cleanup]")

# # -------- OPTIONAL CLEAN MODEL --------
# if CLEANUP_AFTER_INFER:
#     del model, tokenizer, image_processor, context_len
#     cleanup_cuda()
#     show_gpu_mem("[After model cleanup]")
#     print("Model cleaned. Nếu nvidia-smi vẫn còn giữ GPU RAM, restart kernel là sạch nhất.")
# else:
#     print("Model is kept in GPU RAM for faster next inference.")

/venv/llava-med/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/venv/llava-med/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/venv/llava-med/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/venv/llava-med/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._regi

[Before load] allocated=0.00 GB | reserved=0.00 GB


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.14s/it]
Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.18.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.14.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.10.mlp.fc1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.8.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.5.mlp.fc2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.13.self_attn.out_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.14.self_attn.v_proj.bias', 'model.vision_tower.vision_tower

[After load] allocated=14.59 GB | reserved=14.86 GB
Using conv_mode: mistral_instruct
Question: What modality is used to take this image?


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Answer: ive imaging technique that uses a strong magnetic field and radio waves to create detailed images of the body's internal structures.
[After infer temp cleanup] allocated=14.63 GB | reserved=14.89 GB
[After model cleanup] allocated=0.03 GB | reserved=0.03 GB
Model cleaned. Nếu nvidia-smi vẫn còn giữ GPU RAM, restart kernel là sạch nhất.


In [27]:
# =========================
# Best LLaVA-Med inference runner for Jupyter/IPYNB
# Accuracy gần code 2, RAM/VRAM sạch hơn code 1
# =========================

import os
import sys
import gc
import torch
from PIL import Image
import os
import sys

sys.path.append('/workspace/MedVQA-Classification/src')
sys.path.append('/workspace/LLaVA-Med/')
from data.data_prep import PreDataset



# -------- CONFIG --------
LLAVA_MED_PATH = "/workspace/LLaVA-Med"
MODEL_PATH = "microsoft/llava-med-v1.5-mistral-7b"

# Nếu 1 GPU T4 16GB dễ OOM, thử LOAD_4BIT=True.
# Nếu ưu tiên accuracy tối đa và đủ VRAM, để False.
LOAD_4BIT = False
LOAD_8BIT = False

MAX_NEW_TOKENS = 64
DO_SAMPLE = False
TEMPERATURE = 0.0
NUM_BEAMS = 1

# Quan trọng: để giống code 2 hơn
CONV_MODE = "mistral_instruct"
FORCE_SIMPLE_IMAGE_TOKEN = True   # dùng "<image>\nquestion", không dùng <im_start>/<im_end>

ANSWER_ONLY = True

sys.path.insert(0, LLAVA_MED_PATH)

from llava.model.builder import load_pretrained_model
from llava.mm_utils import (
    get_model_name_from_path,
    process_images,
    tokenizer_image_token,
)
from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
)
from llava.conversation import conv_templates, SeparatorStyle

try:
    from llava.utils import disable_torch_init
    disable_torch_init()
except Exception:
    pass


def show_gpu_mem(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"{tag} allocated={allocated:.2f} GB | reserved={reserved:.2f} GB")


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def get_input_device(model):
    """
    Device nên dùng cho input_ids.
    Với model shard bằng accelerate, lấy input embedding device thường an toàn hơn model.device.
    """
    try:
        return model.get_input_embeddings().weight.device
    except Exception:
        try:
            return model.device
        except Exception:
            return next(model.parameters()).device


def get_vision_device(model):
    """
    Device nên dùng cho image tensor.
    Nếu không lấy được vision tower thì fallback về input device.
    """
    try:
        vt = model.get_vision_tower()
        if hasattr(vt, "device"):
            return vt.device
        return next(vt.parameters()).device
    except Exception:
        return get_input_device(model)


def move_image_tensor(image_tensor, device, dtype=torch.float16):
    if isinstance(image_tensor, list):
        return [img.to(device=device, dtype=dtype, non_blocking=True) for img in image_tensor]
    return image_tensor.to(device=device, dtype=dtype, non_blocking=True)


def clean_answer_text(answer, conv):
    """
    Không split USER/ASSISTANT lấy phần trước nữa, vì có thể làm mất answer.
    Chỉ cắt stop string và prefix ở đầu.
    """
    answer = answer.strip()

    # Stop string theo conversation template
    try:
        stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
        if stop_str and stop_str in answer:
            answer = answer.split(stop_str)[0].strip()
    except Exception:
        pass

    # Special tokens thường gặp
    for stop in ["</s>", "###"]:
        if stop in answer:
            answer = answer.split(stop)[0].strip()

    # Nếu model sinh prefix role ở ĐẦU câu thì bỏ prefix, không lấy phần trước prefix
    prefixes = [
        "ASSISTANT:",
        "Assistant:",
        "assistant:",
        "[/INST]",
    ]
    for p in prefixes:
        if answer.startswith(p):
            answer = answer[len(p):].strip()

    return answer.strip()
def detect_question_type(question: str) -> str:
    q = question.lower().strip()

    if q.startswith(("is ", "are ", "does ", "do ", "did ", "can ", "has ", "have ")):
        return "closed"

    if "how many" in q or q.startswith("number of ") or "count" in q:
        return "count"

    if "modality" in q or "image modality" in q or "used to take this image" in q:
        return "modality"

    if any(x in q for x in ["where", "located", "location"]):
        return "location"

    if any(x in q for x in ["organ", "anatomy", "body part", "structure"]):
        return "anatomy"

    return "open"


def build_vqa_prompt(question: str) -> str:
    qtype = detect_question_type(question)

    base = """
        You are a medical visual question answering model for radiology images.

        General rules:
        - Answer using the image and the question.
        - Output only the final answer.
        - Do not explain.
        - Do not repeat the question.
        - Use lowercase.
        """.strip()

    if qtype == "closed":
        extra = 'This is a yes/no question. Answer only "yes" or "no".'
    elif qtype == "count":
        extra = "This is a counting question. Answer only one digit, such as 0, 1, 2, 3."
    elif qtype == "modality":
        extra = 'This is a modality question. Answer only one of: "ct", "mri", "x-ray", "ultrasound", "pet".'
    elif qtype == "location":
        extra = "This is a location question. Answer with a short anatomical location phrase."
    elif qtype == "anatomy":
        extra = "This is an anatomy question. Answer with the shortest correct anatomy or organ name."
    else:
        extra = "This is an open-ended medical VQA question. Answer with a short noun phrase, usually 1 to 5 words."

    return base + "\n" + extra

def build_vqa_prompt(question: str) -> str:
    qtype = detect_question_type(question)

    base = """
        You are a medical visual question answering model for radiology images.

        General rules:
        - Answer using the image and the question.
        - Output only the final answer.
        - Do not explain.
        - Do not repeat the question.
        - Use lowercase.
        """.strip()

    if qtype == "closed":
        extra = 'This is a yes/no question. Answer only "yes" or "no".'
    elif qtype == "count":
        extra = "This is a counting question. Answer only one digit, such as 0, 1, 2, 3."
    elif qtype == "modality":
        extra = 'This is a modality question. Answer only one of: "ct", "mri", "x-ray", "ultrasound", "pet".'
    elif qtype == "location":
        extra = "This is a location question. Answer with a short anatomical location phrase."
    elif qtype == "anatomy":
        extra = "This is an anatomy question. Answer with the shortest correct anatomy or organ name."
    else:
        extra = "This is an open-ended medical VQA question. Answer with a short noun phrase, usually 1 to 5 words."

    return base + "\n" + extra

class LlavaMedRunner:
    def __init__(
        self,
        model_path=MODEL_PATH,
        conv_mode=CONV_MODE,
        load_4bit=LOAD_4BIT,
        load_8bit=LOAD_8BIT,
        force_simple_image_token=FORCE_SIMPLE_IMAGE_TOKEN,
    ):
        self.model_path = model_path
        self.conv_mode = conv_mode
        self.load_4bit = load_4bit
        self.load_8bit = load_8bit
        self.force_simple_image_token = force_simple_image_token

        show_gpu_mem("[Before load]")

        self.model_name = get_model_name_from_path(model_path)

        # Thử load theo API mới trước, nếu repo cũ không hỗ trợ thì fallback.
        try:
            self.tokenizer, self.model, self.image_processor, self.context_len = load_pretrained_model(
                model_path=model_path,
                model_base=None,
                model_name=self.model_name,
                load_4bit=load_4bit,
                load_8bit=load_8bit,
                device_map="auto" if (load_4bit or load_8bit) else None,
            )
        except TypeError:
            self.tokenizer, self.model, self.image_processor, self.context_len = load_pretrained_model(
                model_path=model_path,
                model_base=None,
                model_name=self.model_name,
                load_4bit=load_4bit,
                load_8bit=load_8bit,
            )

        self.model.eval()

        # Nếu conv_mode không tồn tại thì tự fallback
        if self.conv_mode not in conv_templates:
            if "mistral" in self.model_name.lower() and "mistral_instruct" in conv_templates:
                self.conv_mode = "mistral_instruct"
            elif "llava_v1" in conv_templates:
                self.conv_mode = "llava_v1"
            else:
                self.conv_mode = list(conv_templates.keys())[0]

        print("Model name:", self.model_name)
        print("Using conv_mode:", self.conv_mode)
        print("Input device:", get_input_device(self.model))
        print("Vision device:", get_vision_device(self.model))
        show_gpu_mem("[After load]")

    @torch.inference_mode()
    def infer(
        self,
        image,
        question,
        max_new_tokens=MAX_NEW_TOKENS,
        answer_only=ANSWER_ONLY,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE,
        num_beams=NUM_BEAMS,
    ):
        # PIL image
        if not isinstance(image, Image.Image):
            raise TypeError("image phải là PIL.Image.Image")
        image = image.convert("RGB")

        # -------- prompt: mặc định giống code 2 --------
        if self.force_simple_image_token:
            image_token = DEFAULT_IMAGE_TOKEN
        else:
            if getattr(self.model.config, "mm_use_im_start_end", False):
                image_token = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN
            else:
                image_token = DEFAULT_IMAGE_TOKEN

        system_prompt = build_vqa_prompt(question)

        user_question = (
            image_token
            + "\n"
            + system_prompt
            + "\n\nQuestion: "
            + str(question)
            + "\nAnswer:"
        )


        conv = conv_templates[self.conv_mode].copy()
        conv.append_message(conv.roles[0], user_question)
        conv.append_message(conv.roles[1], None)
        prompt = conv.get_prompt()

        input_device = get_input_device(self.model)
        vision_device = get_vision_device(self.model)

        # -------- image preprocess --------
        image_tensor = process_images(
            [image],
            self.image_processor,
            self.model.config,
        )

        image_tensor = move_image_tensor(
            image_tensor,
            device=vision_device,
            dtype=torch.float16,
        )

        image_sizes = [image.size]

        # -------- tokenize --------
        input_ids = tokenizer_image_token(
            prompt,
            self.tokenizer,
            IMAGE_TOKEN_INDEX,
            return_tensors="pt",
        ).unsqueeze(0).to(input_device)
        attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=input_ids.device)

        pad_token_id = self.tokenizer.pad_token_id
        if pad_token_id is None:
            pad_token_id = self.tokenizer.eos_token_id
        gen_kwargs = dict(
            images=image_tensor,
            image_sizes=image_sizes,
            attention_mask=attention_mask,
            do_sample=do_sample,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            num_beams=num_beams,
            pad_token_id=pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        # Khi do_sample=False thì không cần truyền temperature.
        # Truyền temperature=0 đôi khi chỉ warning, không tăng accuracy.
        if do_sample:
            gen_kwargs["temperature"] = temperature

        output_ids = self.model.generate(
            input_ids,
            **gen_kwargs,
        )

        # if answer_only and output_ids.shape[1] > input_ids.shape[1]:
        #     decode_ids = output_ids[0, input_ids.shape[1]:]
        # else:
        decode_ids = output_ids[0]

        answer = self.tokenizer.decode(
            decode_ids,
            skip_special_tokens=True,
        )

        answer = clean_answer_text(answer, conv)

        # Xóa tensor tạm sau mỗi lần infer
        del image_tensor, input_ids, output_ids, decode_ids
        cleanup_cuda()

        return answer

    def unload(self):
        try:
            del self.model
            del self.tokenizer
            del self.image_processor
            del self.context_len
        except Exception:
            pass
        cleanup_cuda()
        show_gpu_mem("[After unload]")


# =========================
# Cách dùng trong notebook
# =========================

# Nếu lỡ chạy cell nhiều lần, unload runner cũ trước khi load model mới
if "runner" in globals():
    try:
        runner.unload()
    except Exception:
        pass

runner = LlavaMedRunner()

t_s=PreDataset('/workspace/MedVQA-Classification/datasets/vqa-rad','test')
index=10
# Infer 1 sample
image = t_s[index]["image"]
question = t_s[index]["question"]

answer = runner.infer(image, question)
print("Question:", question)
print("Answer:", answer)
print("Answer Truth:", t_s[index]["answer"])

# Infer nhiều sample, KHÔNG load lại model
# for i in range(10):
#     ans = runner.infer(t_s[i]["image"], t_s[i]["question"])
#     print(i, t_s[i]["question"], "=>", ans)

# Khi xong hẳn mới gọi:
# runner.unload()

[After unload] allocated=0.03 GB | reserved=0.03 GB
[Before load] allocated=0.03 GB | reserved=0.03 GB


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.20s/it]
Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.encoder.layers.15.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.1.self_attn.v_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.3.self_attn.q_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.19.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.20.self_attn.out_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.3.self_attn.q_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.10.mlp.fc2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.1.mlp.fc1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.23.self_attn.v_proj.weight', 'model.vision_tower.vision_towe

Model name: llava-med-v1.5-mistral-7b
Using conv_mode: mistral_instruct
Input device: cuda:0
Vision device: cuda:0
[After load] allocated=14.63 GB | reserved=14.89 GB
Question: Is there any intraparenchymal abnormalities in the lung fields?
Answer: No, there are no intraparenchymal abnormalities in the lung fields.
Answer Truth: No
